<a href="https://colab.research.google.com/github/fpellerano/devllm/blob/main/20_1_Basic_LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

To upload a `requirements.txt` file to your Google Colab environment, follow these steps:

1.  **Open the Files pane**: Click the **Folder icon** in the left-hand sidebar.
2.  **Upload your file**:
    *   **Drag and drop** your file directly into the pane.
    *   **OR** click the **Upload to session storage** icon (a file with an upward arrow) to select it from your local machine.

  
>[IMPORTANT] Files uploaded this way are temporary and will be deleted once the session is recycled.

# Basic LLMs with LangChain

This notebook is an introduction to interacting with Large Language Models (LLMs) using **LangChain**, one of the most popular frameworks for building LLM-powered applications.

We will cover:
- Setting up and configuring a Chat Model (powered by **Google Gemini**)
- LangChain's core **message types**
- **Direct invocation** of a model
- **Model configuration** (temperature, token limits)
- **Prompt templating** for dynamic, reusable inputs
- **Output parsers** to extract structured data from responses
- **Chaining** components with the LangChain Expression Language (LCEL)
- **Streaming** responses token by token
- **Batch invocations** for processing multiple inputs at once
- **Conversation memory** with LangGraph

In [ ]:
# Installing the necessary libraries
!pip install -qU -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 72.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.1/210.1 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 104.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 74.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0

In [ ]:
# Getting the Gemini API Key
from google.colab import userdata

google_api_key = userdata.get("GOOGLE_API_KEY")

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

# Create a Chat Model interface with Gemini
# - model:       the specific Gemini model variant to use
# - temperature: controls randomness (0 = focused/deterministic, 1 = creative/varied)
model = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite-preview",
    google_api_key=google_api_key,
    temperature=0.7,
)

print(f"Model ready: {model.model}")

Model ready: gemini-3.1-flash-lite-preview


## LangChain Message Types

LangChain standardizes communication with LLMs through a set of **message types**. Each message carries a role that tells the model who is speaking:

| Message Type | Role | Description |
|---|---|---|
| `SystemMessage` | `system` | Sets the behavior, persona, or instructions for the model |
| `HumanMessage` | `user` | A message from the human user |
| `AIMessage` | `assistant` | A response generated by the model |

These map directly to the roles used in most Chat APIs (OpenAI, Gemini, Anthropic, etc.), making it easy to switch providers without changing your application logic.

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

# Each message type wraps a content string and a role
system_msg = SystemMessage(content="You are a helpful assistant that explains things simply.")
human_msg  = HumanMessage(content="What is a neural network?")
ai_msg     = AIMessage(content="A neural network is a system of algorithms inspired by the human brain.")

# Inspect their properties
for msg in [system_msg, human_msg, ai_msg]:
    print(f"Type: {type(msg).__name__:15s} | Role: {msg.type:10s} | Content: {msg.content[:60]}")

Type: SystemMessage   | Role: system     | Content: You are a helpful assistant that explains things simply.
Type: HumanMessage    | Role: human      | Content: What is a neural network?
Type: AIMessage       | Role: ai         | Content: A neural network is a system of algorithms inspired by the h


## Direct Invocation

The most basic way to interact with a Chat Model in LangChain is `.invoke()` — you pass a list of messages and the model returns the next `AIMessage`.

The response object contains not just the text, but also **metadata** like token usage and the model name, which is useful for monitoring costs.

In [ ]:
messages = [
    SystemMessage(content="Translate the following from English into Italian."),
    HumanMessage(content="Hello World! Welcome to the course."),
]

response = model.invoke(messages)
response.pretty_print()

# The response is an AIMessage with rich metadata
print(f"\nToken usage: {response.usage_metadata}")

================================== Ai Message ==================================

[{'type': 'text', 'text': 'Ciao mondo! Benvenuti al corso.', 'extras': {'signature': 'EjQKMgG+Pvb7l8wWQsya9FjkIzzuI37wnp9xX4wDp1hp8a3mgjAjnVMg2hOvTZS7EseLGddx'}}]

Token usage: {'input_tokens': 17, 'output_tokens': 8, 'total_tokens': 25, 'input_token_details': {'cache_read': 0}}


## Model Configuration

LangChain lets you control model behavior through parameters. The two most important ones are:

- **`temperature`**: Controls how random/creative the output is.
  - `0.0` → deterministic, consistent, factual
  - `1.0` → creative, varied, sometimes surprising
- **`max_tokens`**: Hard cap on the number of tokens the model can generate.

You can set these at model creation time, or use **`.bind()`** to attach parameters to any point in a chain without creating a new model instance.

In [ ]:
story_prompt = [
    SystemMessage("You are a creative writer."),
    HumanMessage("Write a one-sentence story about a robot discovering music."),
]

# Temperature 0.0 - focused and repeatable
cold_model = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite-preview",
    google_api_key=google_api_key,
    temperature=0.0,
)
print("=== Temperature 0.0 (Deterministic) ===")
cold_model.invoke(story_prompt).pretty_print()

# Temperature 1.0 - creative and varied
creative_model = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite-preview",
    google_api_key=google_api_key,
    temperature=1.0,
)
print("\n=== Temperature 1.0 (Creative) ===")
creative_model.invoke(story_prompt).pretty_print()

=== Temperature 0.0 (Deterministic) ===
================================== Ai Message ==================================

[{'type': 'text', 'text': 'As the needle dropped, the robot’s rigid processors suddenly dissolved into a symphony of colors it had no name for, and for the first time, its cooling fans hummed in perfect rhythm with the melody.', 'extras': {'signature': 'EjQKMgG+Pvb7iB/pBsXq7CoKYozZm16wwtaPhIbqwKY9QuAimYtzbzCilRrjEybYglw741Vx'}}]

=== Temperature 1.0 (Creative) ===
================================== Ai Message ==================================

[{'type': 'text', 'text': 'As the needle dropped onto the spinning vinyl, the robot felt a strange, rhythmic ghost flicker to life within its cold, metallic circuitry.', 'extras': {'signature': 'EjQKMgG+Pvb7RC8jsLW+5yWFXr2ZCRbUetDdLv3hNiMTOMAzccig8QoXrtxRlcSZPV1Ytoa3'}}]


In [ ]:
limited_model = model.bind(max_output_tokens=15)

print("=== Response limited to 15 tokens ===")
limited_model.invoke([HumanMessage("Explain quantum physics in detail.")]).pretty_print()

=== Response limited to 15 tokens ===
================================== Ai Message ==================================

[{'type': 'text', 'text': 'Quantum physics (or quantum mechanics) is the branch of', 'extras': {'signature': 'EjQKMgG+Pvb7d62T2pN7WbBnYcGUw47jIN3QpKAurMkbcwwdplAvjbEL4JizNUKNhMQg/0kL'}}]


## Prompt Templating

Hardcoding messages is fine for one-off calls, but real applications need **dynamic inputs**. LangChain's `ChatPromptTemplate` lets you define a message structure with named placeholders `{like_this}` that get filled in at runtime.

Benefits:
- **Reusability**: define once, invoke with different data
- **Clarity**: separates the prompt structure from the variable content
- **Composability**: templates are runnables and plug directly into chains

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

system_template = "You are a professional translator. Translate the following text from English into {language}."

prompt_template = ChatPromptTemplate.from_messages([
    ("system", system_template),
    ("user", "{text}"),
])

# Inspect the formatted messages before sending to the model
formatted = prompt_template.invoke({"language": "French", "text": "Good morning! How are you?"})
print("=== Formatted messages ===")
for message in formatted.to_messages():
    message.pretty_print()

=== Formatted messages ===
================================ System Message ================================

You are a professional translator. Translate the following text from English into French.
================================ Human Message =================================

Good morning! How are you?


In [ ]:
# The same template reused across multiple languages
translation_chain = prompt_template | model

languages = ["Spanish", "Japanese", "Arabic", "Portuguese"]
text = "Hello! Welcome to this course on LLM-powered applications."

print(f"Original: {text}\n")
for lang in languages:
    response = translation_chain.invoke({"language": lang, "text": text})
    print(f"{lang:12s}: {response.content}")

Original: Hello! Welcome to this course on LLM-powered applications.

Spanish     : [{'type': 'text', 'text': '¡Hola! Bienvenido a este curso sobre aplicaciones impulsadas por LLM.', 'extras': {'signature': 'EjQKMgG+Pvb7ZDU0gnJExCYvxC2VVR7tZEeNNn1MkPGKnkojEd3Xmxta83K3BB4Tu9lbkr8Z'}}]
Japanese    : [{'type': 'text', 'text': 'こんにちは！LLMを活用したアプリケーション開発コースへようこそ。', 'extras': {'signature': 'EjQKMgG+Pvb79OnYTs1z86MG3DWXSjXSH/+QS0ceevTXA7yr6kpzOw5mrxa/pThA/NvIA6MP'}}]
Arabic      : [{'type': 'text', 'text': 'أهلاً بك! مرحباً بك في هذه الدورة التدريبية حول التطبيقات المعتمدة على النماذج اللغوية الكبيرة (LLMs).', 'extras': {'signature': 'EjQKMgG+Pvb7s3fdNrIoFSwxeLpSQcrOmcS1moyXNuIAo3AIDktpwdux88UL7rabO30/v+sq'}}]
Portuguese  : [{'type': 'text', 'text': 'Olá! Bem-vindo(a) a este curso sobre aplicações baseadas em LLMs.', 'extras': {'signature': 'EjQKMgG+Pvb72//XPDLtUWCvRdOlV/UfQXQc0leLFgpwuuYAliU3EK85bPcFxsIZ7OLEGo+v'}}]


## Output Parsers

By default, LangChain models return an `AIMessage` object. **Output parsers** transform this into more useful formats:

| Parser | Output | Use case |
|---|---|---|
| `StrOutputParser` | `str` | Plain text - the most common parser |
| `JsonOutputParser` | `dict` | Structured data extraction |
| `PydanticOutputParser` | Pydantic model | Validated, typed structured output |

Using output parsers is the **standard pattern** in LangChain chains. A `StrOutputParser` at the end of a chain means downstream components receive a clean string instead of an `AIMessage`.

In [ ]:
from langchain_core.output_parsers import StrOutputParser

# Without a parser: chain returns an AIMessage
chain_no_parser = prompt_template | model
raw = chain_no_parser.invoke({"language": "German", "text": "See you tomorrow!"})
print(f"Without parser -> type: {type(raw).__name__}")
print(f"Content: {raw.content}\n")

# With StrOutputParser: chain returns a plain string
chain_with_parser = prompt_template | model | StrOutputParser()
result = chain_with_parser.invoke({"language": "German", "text": "See you tomorrow!"})
print(f"With StrOutputParser -> type: {type(result).__name__}")
print(f"Content: {result}")

Without parser -> type: AIMessage
Content: [{'type': 'text', 'text': 'Bis morgen!', 'extras': {'signature': 'EjQKMgG+Pvb7ATMQPUQgP3l8R2atLU5yLE+VVblq/XYaLVwlNRbhoY7vTuqgcuujXrasDfVm'}}]

With StrOutputParser -> type: TextAccessor
Content: Bis morgen!


In [ ]:
from langchain_core.output_parsers import JsonOutputParser

# Instruct the model to return JSON and parse it automatically
json_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Always respond with valid JSON only, no markdown fences."),
    ("user", "Give me key facts about {country}. Include: name (str), capital (str), population (int), continent (str), and official_language (str)."),
])

json_chain = json_prompt | model | JsonOutputParser()

for country in ["Brazil", "Japan", "Nigeria"]:
    result = json_chain.invoke({"country": country})
    print(f"{result['name']:10s} | Capital: {result['capital']:15s} | Population: {result['population']:,}")

Brazil     | Capital: Brasília        | Population: 215,313,498
Japan      | Capital: Tokyo           | Population: 125,100,000
Nigeria    | Capital: Abuja           | Population: 218,541,212


## Chaining with LCEL

The **LangChain Expression Language (LCEL)** uses the `|` pipe operator to compose components into a **chain**. Each component's output becomes the next component's input:

```
prompt_template | model | output_parser
```

LCEL chains are:
- **Lazy**: nothing executes until `.invoke()` (or `.stream()` / `.batch()`) is called
- **Composable**: any `Runnable` can be piped into another
- **Observable**: attach callbacks for logging and tracing at any step
- **Streaming-native**: streaming works out of the box on all LCEL chains

The `ConsoleCallbackHandler` is a handy debug tool that prints a trace of each step as the chain executes.

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.tracers.stdout import ConsoleCallbackHandler

summarize_template = ChatPromptTemplate.from_messages([
    ("system", "You are an expert at summarizing text. Be concise and accurate."),
    ("user", "Summarize the following text in exactly {num_sentences} sentence(s):\n\n{text}"),
])

summarize_chain = summarize_template | model | StrOutputParser()

long_text = """
LangChain is an open-source framework designed to simplify the creation of applications
using large language models (LLMs). It provides a standard interface for interacting with
dozens of LLM providers, a collection of useful integrations (vector stores, document loaders,
tools), and patterns for building, debugging, and deploying LLM-powered applications.
LangChain supports chains, agents, and retrieval-augmented generation (RAG) workflows,
making it versatile for use cases ranging from simple chatbots to complex autonomous agents.
"""

print("=== 1-sentence summary ===")
print(summarize_chain.invoke({"num_sentences": 1, "text": long_text}))

print("\n=== 2-sentence summary ===")
print(summarize_chain.invoke({"num_sentences": 2, "text": long_text}))

=== 1-sentence summary ===
LangChain is an open-source framework that simplifies the development of LLM-powered applications by providing a unified interface, diverse integrations, and robust tools for building complex workflows like agents and RAG.

=== 2-sentence summary ===
LangChain is an open-source framework that simplifies the development of applications powered by large language models through a unified interface and extensive integration tools. It supports versatile workflows like RAG and autonomous agents, enabling the creation of systems ranging from basic chatbots to complex AI solutions.


In [ ]:
# The ConsoleCallbackHandler prints a trace of every step in the chain
print("=== Chain execution with tracing ===")
result = summarize_chain.invoke(
    {"num_sentences": 1, "text": long_text},
    config={"callbacks": [ConsoleCallbackHandler()]},
)

=== Chain execution with tracing ===
[chain/start] [chain:RunnableSequence] Entering Chain run with input:
{
  "num_sentences": 1,
  "text": "\nLangChain is an open-source framework designed to simplify the creation of applications\nusing large language models (LLMs). It provides a standard interface for interacting with\ndozens of LLM providers, a collection of useful integrations (vector stores, document loaders,\ntools), and patterns for building, debugging, and deploying LLM-powered applications.\nLangChain supports chains, agents, and retrieval-augmented generation (RAG) workflows,\nmaking it versatile for use cases ranging from simple chatbots to complex autonomous agents.\n"
}
[chain/start] [chain:RunnableSequence > prompt:ChatPromptTemplate] Entering Prompt run with input:
{
  "num_sentences": 1,
  "text": "\nLangChain is an open-source framework designed to simplify the creation of applications\nusing large language models (LLMs). It provides a standard interface for interacti

## Streaming

Instead of waiting for the complete response, LangChain supports **streaming** - receiving tokens as they are generated by the model. This is important for:

- **Perceived latency**: users see output immediately rather than after a long wait
- **Real-time processing**: downstream components can start working on partial output
- **Responsive UIs**: the foundation of chat interfaces that feel instant

Use `.stream()` instead of `.invoke()` to get a generator of chunks. All LCEL chains support streaming natively.

In [ ]:
stream_chain = summarize_template | model | StrOutputParser()

print("Streaming response (tokens arrive progressively):\n")
for chunk in stream_chain.stream({"num_sentences": 3, "text": long_text}):
    print(chunk, end="", flush=True)

print("\n\n--- Stream complete ---")

Streaming response (tokens arrive progressively):

LangChain is an open-source framework that streamlines the development of applications powered by large language models. It offers a standardized interface, extensive integrations, and specialized patterns for building, debugging, and deploying AI solutions. By supporting chains, agents, and RAG workflows, the framework enables the creation of diverse tools ranging from basic chatbots to complex autonomous systems.

--- Stream complete ---


## Batch Invocation

When you need to process **multiple inputs**, `.batch()` is more efficient than calling `.invoke()` in a loop. LangChain can execute the calls concurrently under the hood, significantly reducing total wall-clock time.

`.batch()` takes a list of input dicts and returns a list of results in the same order.

In [ ]:
import time

translation_chain = prompt_template | model | StrOutputParser()

inputs = [
    {"language": "Spanish",    "text": "Good morning!"},
    {"language": "French",     "text": "Good morning!"},
    {"language": "German",     "text": "Good morning!"},
    {"language": "Japanese",   "text": "Good morning!"},
    {"language": "Portuguese", "text": "Good morning!"},
]

start = time.time()
results = translation_chain.batch(inputs)
elapsed = time.time() - start

print(f"Processed {len(inputs)} translations in {elapsed:.2f}s\n")
for inp, result in zip(inputs, results):
    print(f"{inp['language']:12s}: {result}")

Processed 5 translations in 1.31s

Spanish     : ¡Buenos días!
French      : Bonjour !
German      : Guten Morgen!
Japanese    : おはようございます。
Portuguese  : Bom dia!


## Conversation Memory

A single `.invoke()` call is **stateless** - the model has no memory of previous turns. To build a multi-turn conversation, you must maintain and send the full message history with each call.

**LangGraph** makes this straightforward. It lets you define a stateful graph where:
- The **state** holds the conversation history and any additional context
- A **checkpointer** (e.g. `MemorySaver`) persists state between calls, identified by a `thread_id`
- Each call automatically picks up right where the last one left off

Multiple independent conversations can run in parallel, each with a unique `thread_id`.

In [ ]:
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import BaseMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import START, StateGraph
from langgraph.graph.message import add_messages
from typing_extensions import Annotated, TypedDict


class State(TypedDict):
    """
    Represents the state of the conversation at any point in time.

    Parameters
    ==========
    language: str
        The language in which the LLM should respond.
    messages: list[BaseMessage]
        The full back-and-forth message history. The `add_messages` reducer
        appends new messages rather than overwriting the list.
    """
    language: str
    messages: Annotated[list[BaseMessage], add_messages]


system_template = "You will answer all questions in {language} regardless of the language used in the query."
prompt_template = ChatPromptTemplate.from_messages([
    ("system", system_template),
    MessagesPlaceholder(variable_name="messages"),
])


def call_model(state: State):
    chain = prompt_template | model
    response = chain.invoke(state)
    return {"messages": [response]}


workflow = StateGraph(state_schema=State)
workflow.add_edge(START, "model")
workflow.add_node("model", call_model)

memory = MemorySaver()
app = workflow.compile(checkpointer=memory)

print("Stateful conversation graph compiled successfully.")

Stateful conversation graph compiled successfully.


In [ ]:
config = {"configurable": {"thread_id": "intro_thread_1"}}

# Turn 1: introduce ourselves
print("=== Turn 1 ===")
output = app.invoke(
    {"messages": [HumanMessage("Hey! My name is Alex and I'm learning about LLMs.")], "language": "English"},
    config,
)
output["messages"][-1].pretty_print()

=== Turn 1 ===
================================== Ai Message ==================================

[{'type': 'text', 'text': 'Hi Alex! It’s great to meet you. Welcome to the world of Large Language Models (LLMs)—it is an incredibly fast-moving and exciting field to dive into.\n\nSince you\'re just starting out, is there a specific area you\'re curious about? For example:\n\n*   **The Basics:** How do they actually "think" or predict the next word? (Tokens, Transformers, Neural Networks).\n*   **Practical Use:** How to get the best results out of them (Prompt Engineering).\n*   **The Ecosystem:** What’s the difference between models like GPT-4, Claude, Llama, and Gemini?\n*   **Development:** How to build apps using LLMs (APIs, RAG, LangChain).\n\nFeel free to ask me anything—no question is too simple! Where would you like to start?', 'extras': {'signature': 'EjQKMgG+Pvb7YdBJqxjWDJGwibQqLcRrK3sHzLoduHvVMh2+nJYM6YAsAIW1A758KjXDsXuN'}}]


In [ ]:
# Turn 2: ask an unrelated question - the model should still remember our name
print("=== Turn 2 ===")
output = app.invoke(
    {"messages": [HumanMessage("What are the main differences between GPT and BERT?")]},
    config,
)
output["messages"][-1].pretty_print()

=== Turn 2 ===
================================== Ai Message ==================================

[{'type': 'text', 'text': 'That is a fantastic question! It hits on the two fundamental "architectures" that shaped the modern AI landscape.\n\nTo understand the difference, you first need to know that both GPT and BERT are built on the **Transformer** architecture (introduced in the famous 2017 paper *"Attention Is All You Need"*), but they use different parts of that architecture to achieve different goals.\n\nHere is the breakdown:\n\n### 1. The Core Architecture\n*   **GPT (Generative Pre-trained Transformer):** Uses only the **Decoder** part of the Transformer. It is built to be a "causal" or "autoregressive" model.\n*   **BERT (Bidirectional Encoder Representations from Transformers):** Uses only the **Encoder** part of the Transformer.\n\n### 2. How They "Read" (Directionality)\nThis is the most important technical difference:\n*   **GPT is Unidirectional:** When GPT predicts the nex

In [ ]:
# Turn 3: test memory - does it remember the name from Turn 1?
print("=== Turn 3 ===")
output = app.invoke(
    {"messages": [HumanMessage("Do you remember what my name is?")]},
    config,
)
output["messages"][-1].pretty_print()

=== Turn 3 ===
================================== Ai Message ==================================

[{'type': 'text', 'text': 'Yes, I do! Your name is **Alex**.', 'extras': {'signature': 'EjQKMgG+Pvb72XPQMf0a6FWiRLgpDeIqjw58/5gtunAT7Q/PgrX9ANXoDOggso55c+Rs3nBH'}}]


In [ ]:
# Inspect the full conversation history stored by MemorySaver
all_messages = output["messages"]
print(f"Total messages in thread: {len(all_messages)}\n")

for i, msg in enumerate(all_messages):
    role = type(msg).__name__

    # Extract the actual text content from the message
    actual_text_content = ""
    if isinstance(msg.content, str):
        actual_text_content = msg.content
    elif isinstance(msg.content, list):
        extracted_texts = []
        for item in msg.content:
            if isinstance(item, dict) and 'text' in item:
                extracted_texts.append(item['text'])
            elif isinstance(item, str): # Handle cases where list might contain strings directly
                extracted_texts.append(item)
        actual_text_content = " ".join(extracted_texts)
    else:
        # Fallback for unexpected content types, just convert to string
        actual_text_content = str(msg.content)

    preview = actual_text_content[:90].replace("\n", " ")
    ellipsis = "..." if len(actual_text_content) > 90 else ""
    print(f"[{i}] {role:15s}: {preview}{ellipsis}")

Total messages in thread: 6

[0] HumanMessage   : Hey! My name is Alex and I'm learning about LLMs.
[1] AIMessage      : Hi Alex! It’s great to meet you. Welcome to the world of Large Language Models (LLMs)—it i...
[2] HumanMessage   : What are the main differences between GPT and BERT?
[3] AIMessage      : That is a fantastic question! It hits on the two fundamental "architectures" that shaped t...
[4] HumanMessage   : Do you remember what my name is?
[5] AIMessage      : Yes, I do! Your name is **Alex**.


In [ ]:
# A new thread_id starts a completely fresh conversation with no prior memory
fresh_config = {"configurable": {"thread_id": "intro_thread_2"}}

print("=== Fresh thread - no prior context ===")
output = app.invoke(
    {"messages": [HumanMessage("Do you remember what my name is?")], "language": "English"},
    fresh_config,
)
output["messages"][-1].pretty_print()

=== Fresh thread - no prior context ===
================================== Ai Message ==================================

[{'type': 'text', 'text': "I do not have access to your personal information, past conversations, or any data that would identify you. Each session starts with a clean slate, so I don't know your name unless you have shared it earlier in this specific conversation.", 'extras': {'signature': 'EjQKMgG+Pvb7xeFOHLCGB3XqrIsqX2v1a7J5S2bMduEYEJd3hISkmZ58WPmqylxLvuYGXtoq'}}]


## References

* [LangChain Documentation](https://python.langchain.com/docs/)
* [LangChain Expression Language (LCEL)](https://python.langchain.com/docs/concepts/lcel/)
* [Chat Models - LangChain Concepts](https://python.langchain.com/docs/concepts/chat_models/)
* [Prompt Templates](https://python.langchain.com/docs/concepts/prompt_templates/)
* [Output Parsers](https://python.langchain.com/docs/concepts/output_parsers/)
* [Build a Chatbot](https://python.langchain.com/docs/tutorials/chatbot/)
* [LangGraph Glossary](https://langchain-ai.github.io/langgraph/concepts/low_level/)
* [LangChain Google GenAI Integration](https://python.langchain.com/docs/integrations/chat/google_generative_ai/)